# Ogham OCR Training Notebook

This notebook provides a complete training pipeline for fine-tuning TrOCR on Ogham inscriptions.

## Setup

1. Mount Google Drive
2. Install dependencies
3. Clone repository
4. Configure training

In [ ]:
# Install dependencies
!pip install -q transformers torch albumentations wandb editdistance Pillow opencv-python tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repository (update with your actual repo URL)
# If running locally, skip this cell — just cd to your project directory
import os
if os.path.exists('/content'):
    # Running on Colab
    !git clone https://github.com/daranewsome/ogham_ocr.git
    %cd ogham_ocr
else:
    print("Running locally — ensure you're in the ogham_ocr project directory")

In [ ]:
# Add to Python path
import sys
sys.path.insert(0, '/content/ogham_ocr')

## Configuration

In [ ]:
from src.training import TrainingConfig
from src.training.colab_storage import ColabStorageManager

# Initialize storage manager
storage = ColabStorageManager()

# Training configuration
config = TrainingConfig(
    experiment_name="ogham_ocr_v1",
    
    # Model
    model_name="microsoft/trocr-base-printed",
    max_length=32,
    
    # Training
    num_epochs=50,
    batch_size=16,  # Adjust based on GPU memory
    learning_rate=5e-5,
    
    # Data
    synthetic_size=50000,
    curriculum_schedule="default",
    
    # Checkpointing
    checkpoint_dir=str(storage.get_checkpoint_path("ogham_ocr_v1")),
    
    # Logging
    use_wandb=True,
)

print(f"Config: {config.to_dict()}")

## Load Model

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# Load pre-trained TrOCR
processor = TrOCRProcessor.from_pretrained(config.model_name)
model = VisionEncoderDecoderModel.from_pretrained(config.model_name)

# Configure for Ogham
# Note: For production, you'd want to add Ogham characters to the tokenizer vocabulary
# This is a simplified version that works with the base tokenizer

print(f"Model loaded: {config.model_name}")
print(f"Vocab size: {processor.tokenizer.vocab_size}")

## Create Datasets

In [ ]:
from pathlib import Path
from src.datasets import RealOghamDataset, MixedOghamDataset
from src.datasets.synthetic_dataset import LazySyntheticDataset
from src.datasets.mixed_dataset import CurriculumScheduler
import os

# Determine paths based on environment
if os.path.exists('/content'):
    # Colab: use ColabStorageManager
    data_dir = storage.get_dataset_path("real")
    font_dir = storage.get_font_path()
else:
    # Local: use project-relative paths
    data_dir = Path("./ogham_dataset")
    font_dir = Path("./assets/fonts")

# Check if we have real data
has_real_data = (data_dir / "splits" / "train_stones.txt").exists()

if has_real_data:
    print("Loading real dataset...")
    real_train = RealOghamDataset(
        data_dir=str(data_dir),
        split="train",
        processor=processor,
    )
    real_val = RealOghamDataset(
        data_dir=str(data_dir),
        split="val",
        processor=processor,
    )
    print(f"Real train samples: {len(real_train)}")
    print(f"Real val samples: {len(real_val)}")
else:
    print("No real data found - using synthetic only")
    real_train = None
    real_val = None

# Check for fonts
font_paths = list(font_dir.glob("*.ttf")) + list(font_dir.glob("*.otf"))
print(f"Found {len(font_paths)} font files")

if len(font_paths) == 0:
    print("\n⚠️ No fonts found! Please add Ogham fonts to:", font_dir)
    print("Recommended: BabelStone Ogham, Noto Sans Ogham")

In [ ]:
# Create synthetic dataset (passes processor for consistent normalization with real data)
print("Creating synthetic dataset...")

synthetic_train = LazySyntheticDataset(
    size=config.synthetic_size,
    font_dir=str(font_dir),
    tokenizer=processor.tokenizer,
    processor=processor,
    difficulty="easy",  # Start easy for curriculum
    seed=42,
)

print(f"Synthetic train samples: {len(synthetic_train)}")

In [ ]:
# Create mixed dataset with curriculum learning
curriculum_schedule = CurriculumScheduler.get_default_schedule(config.num_epochs)

if real_train:
    train_dataset = MixedOghamDataset(
        real_dataset=real_train,
        synthetic_dataset=synthetic_train,
        synthetic_ratio=0.95,  # Start with mostly synthetic
        curriculum_schedule=curriculum_schedule,
    )
else:
    # Synthetic only
    train_dataset = synthetic_train

val_dataset = real_val if real_val else synthetic_train

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

## Initialize Training

In [ ]:
from src.training import OghamTrainer, CheckpointManager
from src.evaluation import ExperimentLogger

# Create checkpoint manager
checkpoint_manager = CheckpointManager(
    experiment_name=config.experiment_name,
    checkpoint_dir=config.checkpoint_dir,
)

# Create logger
logger = ExperimentLogger(
    experiment_name=config.experiment_name,
    log_dir=str(storage.get_log_path(config.experiment_name)),
    use_wandb=config.use_wandb,
    config=config.to_dict(),
)

# Create trainer
trainer = OghamTrainer(
    model=model,
    processor=processor,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=config,
    checkpoint_manager=checkpoint_manager,
    logger=logger,
)

print("Trainer initialized!")

## Train!

In [ ]:
# Resume from checkpoint if available
start_epoch = trainer.resume_if_possible()
print(f"Starting from epoch {start_epoch}")

# Train
history = trainer.train(start_epoch=start_epoch)

## Evaluation

In [ ]:
# Final evaluation
print("\nFinal Evaluation:")
final_metrics = trainer.evaluate()

for metric, value in final_metrics.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Finish logging
logger.finish()

print("\n✅ Training complete!")
print(f"Best checkpoint saved to: {config.checkpoint_dir}/best.pt")